Import libraries:

In [ ]:
pip install openai python-dotenv numpy

Test embeddings

In [ ]:
import os
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv

# 1. Load configuration
# load_dotenv() will look for a .env file. 
# In K8s, these are already in os.environ, so this remains safe.
load_dotenv()

API_KEY = os.getenv("EMBEDDING_API_KEY", "empty")
BASE_URL = os.getenv("EMBEDDING_API_BASE_URL")
MODEL_NAME = os.getenv("EMBEDDING_MODEL_NAME", "BAAI/bge-m3")

# 2. Initialize the OpenAI Client
# We point it to the vLLM server instead of OpenAI's official servers
client = OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL
)

def get_embeddings(text_list):
    """
    Sends a list of strings to the vLLM server and returns embeddings.
    """
    try:
        response = client.embeddings.create(
            input=text_list,
            model=MODEL_NAME
        )
        # Extract the embedding vectors from the response object
        return [data.embedding for data in response.data]
    except Exception as e:
        print(f"Error calling the API: {e}")
        return None

def cosine_similarity(a, b):
    """Helper to check if the vectors actually work."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# 3. Execution and Testing
if __name__ == "__main__":
    sentences = [
        "How is the weather today?",
        "What is the current atmospheric condition?",
        "I like eating apples."
    ]

    print(f"Testing model: {MODEL_NAME} at {BASE_URL}...")
    vectors = get_embeddings(sentences)

    if vectors:
        print(f"Successfully retrieved {len(vectors)} vectors.")
        print(f"Vector dimension: {len(vectors[0])}") # BGE-M3 is typically 1024
        
        sim_related = cosine_similarity(vectors[0], vectors[1])
        sim_unrelated = cosine_similarity(vectors[0], vectors[2])
        
        print(f"\nSimilarity (Related): {sim_related:.4f}")
        print(f"Similarity (Unrelated): {sim_unrelated:.4f}")